# Day 008 | File I/O — Read & Write CSV, TXT

## 100 Days of ML for Geotechnical Engineering
**Phase:** Python Fundamentals | **Author:** Ripon Chandra Malo | **Date:** February 23, 2026

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| `open()` + `with` | Open a file safely | `with open("data.csv") as f:` |
| File modes | `"r"` read, `"w"` write, `"a"` append | Read field CSV, write report |
| `.read()` / `.readlines()` | Read entire file or line-by-line | Load borehole data |
| `.write()` | Write text to a file | Save analysis report |
| `csv.reader` / `csv.writer` | Read/write CSV safely | Handle commas in soil names |
| `csv.DictReader` / `DictWriter` | CSV ↔ list of dictionaries | The Pandas-ready pattern |
| Append mode `"a"` | Add to end of file | Drilling log entries |
| `os.path.exists()` | Check if file exists | Safe file reading |

---

## Our Geotechnical Problem

Until now, all our data was **hardcoded** inside the script:

```python
n_values = [5, 8, 12, 18, 28, 35, 50, 60]  # typed by hand!
```

Real projects work with **files**:

```
INPUT:   borehole_log.csv  →  [Your Python Script]  →  OUTPUT: analysis_report.csv
         (from driller)         (processes data)              (for client)
```

Today we learn to **read** data from files and **write** results back — the bridge between your code and the outside world.

---

In [ ]:
import os
import csv

# Create a folder for our data files
os.makedirs("geotech_data", exist_ok=True)
print("Ready! Working directory: geotech_data/")

## Part 1: Writing Text Files — The `with` Statement

To write a file, use `open(filename, mode)` with mode `"w"` (write).

**Always** use the `with` statement — it automatically closes the file, even if an error occurs.

```python
# BAD — must remember to close
f = open("file.txt", "w")
f.write("data")
f.close()              # Forget this → data loss!

# GOOD — auto-closes when 'with' block ends
with open("file.txt", "w") as f:
    f.write("data")
# File automatically closed here ✓
```

### File Modes

| Mode | Meaning | File exists? | File doesn't exist? |
|------|---------|-------------|--------------------|
| `"r"` | Read | Opens it | Error! |
| `"w"` | Write | **Overwrites!** | Creates new |
| `"a"` | Append | Adds to end | Creates new |

⚠️ **Warning:** `"w"` mode **erases** the existing file! Use `"a"` to add without erasing.

In [ ]:
# ─────────────────────────────────────────────────────
# WRITING A TEXT FILE — Field note
# ─────────────────────────────────────────────────────

with open("geotech_data/field_note.txt", "w") as f:
    f.write("Borehole: BH-01\n")           # \n = new line
    f.write("Location: Salt Lake City, Utah\n")
    f.write("Date: 2026-02-23\n")
    f.write("Driller: ABC Drilling Co.\n")
    f.write("Water table: 3.2m\n")
    f.write("Equipment: Automatic trip hammer (60% efficiency)\n")

print("✓ Wrote field_note.txt")
print("  File auto-closed when 'with' block ended.")

---

## Part 2: Reading Text Files

Three ways to read:

| Method | Returns | Best For |
|--------|---------|----------|
| `.read()` | Entire file as ONE string | Small files |
| `.readlines()` | List of ALL lines | When you need all lines in memory |
| `for line in f:` | One line at a time | **Large files** (memory efficient) |

In [ ]:
# ─────────────────────────────────────────────────────
# METHOD 1: .read() — entire file as one string
# ─────────────────────────────────────────────────────

with open("geotech_data/field_note.txt", "r") as f:
    content = f.read()

print("Entire file (.read()):")
print(content)
print(f"Type: {type(content)}")
print(f"Length: {len(content)} characters")

In [ ]:
# ─────────────────────────────────────────────────────
# METHOD 2: .readlines() — list of all lines
# ─────────────────────────────────────────────────────

with open("geotech_data/field_note.txt", "r") as f:
    lines = f.readlines()

print(f"Loaded {len(lines)} lines:")
for i, line in enumerate(lines):
    print(f"  [{i}] '{line.strip()}'")

# Notice: each line still has \n at the end
# Use .strip() to remove it

In [ ]:
# ─────────────────────────────────────────────────────
# METHOD 3: for loop — one line at a time (BEST)
# ─────────────────────────────────────────────────────
# Doesn't load the whole file into memory.
# Perfect for files with millions of rows.

print("Line-by-line reading:")
with open("geotech_data/field_note.txt", "r") as f:
    for line_num, line in enumerate(f, start=1):
        print(f"  Line {line_num}: {line.strip()}")

print()
print("Tip: Use the for-loop method for large data files.")

---

## Part 3: Writing CSV Files — The Universal Data Format

**CSV** (Comma-Separated Values) is the most common format for exchanging tabular data. Excel, Pandas, databases — everything reads CSV.

```
Depth_m,N60,Soil_Type,Vs_ms
1.5,5,Fill,160.8
3.0,8,Silty Sand,186.4
```

Two ways to write CSV:

| Method | When to Use |
|--------|------------|
| Manual `f.write()` | Simple data, no commas in values |
| `csv.writer` | **Recommended** — safely handles commas, quotes |

In [ ]:
# ─────────────────────────────────────────────────────
# WRITING CSV — Manual method
# ─────────────────────────────────────────────────────

depths =     [1.5, 3.0, 4.5, 6.0, 7.5, 9.0, 10.5, 12.0]
n_values =   [5, 8, 12, 18, 28, 35, 50, 60]
soil_types = ["Fill", "Silty Sand", "Sand", "Clayey Sand",
              "Sand", "Gravel", "Weathered Rock", "Bedrock"]

with open("geotech_data/bh01_manual.csv", "w") as f:
    f.write("Depth_m,N60,Soil_Type\n")  # Header
    for d, n, s in zip(depths, n_values, soil_types):
        f.write(f"{d},{n},{s}\n")       # Data rows

print("✓ Wrote bh01_manual.csv")

# Verify
with open("geotech_data/bh01_manual.csv", "r") as f:
    print(f.read())

In [ ]:
# ─────────────────────────────────────────────────────
# WRITING CSV — csv.writer (RECOMMENDED)
# ─────────────────────────────────────────────────────
# Handles commas in values, special characters, etc.
# Always use newline="" on Windows to prevent blank rows.

with open("geotech_data/bh01_spt.csv", "w", newline="") as f:
    writer = csv.writer(f)
    
    # Write header
    writer.writerow(["Depth_m", "N60", "Soil_Type", "Vs_ms"])
    
    # Write each data row
    for d, n, s in zip(depths, n_values, soil_types):
        Vs = round(97 * (n ** 0.314), 1)
        writer.writerow([d, n, s, Vs])

print("✓ Wrote bh01_spt.csv using csv.writer")

with open("geotech_data/bh01_spt.csv", "r") as f:
    print(f.read())

---

## Part 4: Reading CSV Files

Three methods, from basic to best:

| Method | Returns | Best For |
|--------|---------|----------|
| Manual `split(",")` | Raw strings | Understanding how CSV works |
| `csv.reader` | Lists of strings | General CSV reading |
| `csv.DictReader` | **List of dicts** | **Best — ready for Pandas** |

In [ ]:
# ─────────────────────────────────────────────────────
# READING CSV — Manual split (to understand how it works)
# ─────────────────────────────────────────────────────

with open("geotech_data/bh01_spt.csv", "r") as f:
    header = f.readline().strip().split(",")
    print(f"Header: {header}")
    
    data = []
    for line in f:
        parts = line.strip().split(",")
        data.append(parts)

print(f"Rows: {len(data)}")
print(f"Row 0: {data[0]}")
print(f"Row 0 depth: {data[0][0]} (it's a STRING, not a number!)")
print()
print("Manual parsing works but everything is a string.")
print("You must convert: float(data[0][0]) → 1.5")

In [ ]:
# ─────────────────────────────────────────────────────
# READING CSV — csv.DictReader (THE BEST WAY)
# ─────────────────────────────────────────────────────
# Each row becomes a DICTIONARY with column names as keys.
# This is the same pattern as Day 006: list of dicts!

with open("geotech_data/bh01_spt.csv", "r") as f:
    reader = csv.DictReader(f)
    layers = list(reader)

print(f"Loaded {len(layers)} layers")
print(f"Row 0: {layers[0]}")
print()

# Access by column NAME — just like Day 006
print(f"Depth: {layers[0]['Depth_m']}")
print(f"N60:   {layers[0]['N60']}")
print(f"Soil:  {layers[0]['Soil_Type']}")
print()

# IMPORTANT: All values are STRINGS!
# Convert to numbers when needed:
for layer in layers:
    layer['Depth_m'] = float(layer['Depth_m'])
    layer['N60'] = int(layer['N60'])
    layer['Vs_ms'] = float(layer['Vs_ms'])

print("After conversion:")
print(f"  Depth: {layers[0]['Depth_m']} (type: {type(layers[0]['Depth_m']).__name__})")
print(f"  N60:   {layers[0]['N60']} (type: {type(layers[0]['N60']).__name__})")
print()

# Now we can do math!
all_n = [l['N60'] for l in layers]
print(f"Average N60: {sum(all_n)/len(all_n):.1f}")

---

## Part 5: Appending to Files

Mode `"a"` adds to the **end** of a file without erasing existing content. Perfect for logs and ongoing data collection.

In [ ]:
# ─────────────────────────────────────────────────────
# APPEND MODE — Add entries over time
# ─────────────────────────────────────────────────────

# Create the log
with open("geotech_data/drilling_log.txt", "w") as f:
    f.write("DRILLING LOG — BH-01\n")
    f.write("=" * 40 + "\n")

# Morning entries (APPEND — adds to end)
with open("geotech_data/drilling_log.txt", "a") as f:
    f.write("08:00 — Started at 0.0m\n")
    f.write("09:30 — SPT at 1.5m: N=5, Fill\n")
    f.write("10:45 — SPT at 3.0m: N=8, Sand\n")

# Afternoon entries (still APPEND)
with open("geotech_data/drilling_log.txt", "a") as f:
    f.write("13:00 — SPT at 4.5m: N=12, Sand\n")
    f.write("14:15 — Water table at 3.2m\n")
    f.write("16:00 — End of day, depth: 6.0m\n")

# Read the full log
with open("geotech_data/drilling_log.txt", "r") as f:
    print(f.read())

---

## Part 6: DictWriter — Write Dicts Directly to CSV

When your data is already a **list of dictionaries** (the pattern from Day 006), `DictWriter` writes it to CSV with zero manual formatting.

In [ ]:
# ─────────────────────────────────────────────────────
# csv.DictWriter — List of dicts → CSV in 4 lines
# ─────────────────────────────────────────────────────

processed = [
    {"depth": 1.5, "N60": 5, "soil": "Fill", "Vs": 160.8, "class": "Loose"},
    {"depth": 3.0, "N60": 8, "soil": "Silty Sand", "Vs": 186.4, "class": "Loose"},
    {"depth": 4.5, "N60": 12, "soil": "Sand", "Vs": 211.7, "class": "Med Dense"},
    {"depth": 6.0, "N60": 18, "soil": "Clayey Sand", "Vs": 240.4, "class": "Med Dense"},
    {"depth": 7.5, "N60": 28, "soil": "Sand", "Vs": 276.2, "class": "Med Dense"},
    {"depth": 9.0, "N60": 35, "soil": "Gravel", "Vs": 296.2, "class": "Dense"},
]

fieldnames = ["depth", "N60", "soil", "Vs", "class"]

with open("geotech_data/processed_bh01.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()           # Column names
    writer.writerows(processed)    # ALL rows at once

print("✓ List of dicts → CSV in 4 lines!")
with open("geotech_data/processed_bh01.csv", "r") as f:
    print(f.read())

print("In Phase 2, this becomes: pd.DataFrame(processed).to_csv(...)")
print("Same concept, same data structure — Pandas just wraps it.")

---

## Part 7: Complete Application — Field Data Pipeline

Let's build the **full workflow** an engineer would use:

```
raw_field_data.csv  →  [Process]  →  analysis_results.csv
(from driller)          (Python)      (for client)
                           ↓
                   summary_report.txt
                      (for archive)
```

In [ ]:
# ══════════════════════════════════════════════════════
# STEP 1: Create raw field data CSV (simulates driller's file)
# ══════════════════════════════════════════════════════

with open("geotech_data/raw_field_data.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["borehole", "depth_m", "N_raw", "soil_description"])
    writer.writerow(["BH-01", "1.5", "5", "Fill (dark brown)"])
    writer.writerow(["BH-01", "3.0", "8", "Silty SAND (SM)"])
    writer.writerow(["BH-01", "4.5", "12", "SAND (SP)"])
    writer.writerow(["BH-01", "6.0", "18", "clayey Sand (SC)"])
    writer.writerow(["BH-01", "7.5", "28", "SAND (SW)"])
    writer.writerow(["BH-01", "9.0", "35", "Gravel (GP)"])
    writer.writerow(["BH-01", "10.5", "50", "weathered ROCK"])
    writer.writerow(["BH-01", "12.0", "60", "BEDROCK"])

print("Step 1: ✓ Created raw_field_data.csv")

In [ ]:
# ══════════════════════════════════════════════════════
# STEP 2: Read raw data
# ══════════════════════════════════════════════════════

with open("geotech_data/raw_field_data.csv", "r") as f:
    reader = csv.DictReader(f)
    raw_data = list(reader)

print(f"Step 2: ✓ Loaded {len(raw_data)} rows")
print(f"  Sample: {raw_data[0]}")

In [ ]:
# ══════════════════════════════════════════════════════
# STEP 3: Process each row (correct, classify, compute)
# ══════════════════════════════════════════════════════

wt = 3.2
processed = []

for row in raw_data:
    depth = float(row["depth_m"])
    N60 = int(row["N_raw"]) * (60 / 60)
    soil = row["soil_description"].strip().title()
    Vs = 97 * (N60 ** 0.314)
    
    if N60 < 4: cons = "Very Loose"
    elif N60 < 10: cons = "Loose"
    elif N60 < 30: cons = "Medium Dense"
    elif N60 < 50: cons = "Dense"
    else: cons = "Very Dense"
    
    processed.append({
        "borehole": row["borehole"],
        "depth_m": depth, "N60": int(N60),
        "Vs_ms": round(Vs, 1), "soil": soil,
        "consistency": cons,
        "saturated": "Y" if depth > wt else "N",
    })

print(f"Step 3: ✓ Processed {len(processed)} layers")
print(f"  Sample: {processed[0]}")

In [ ]:
# ══════════════════════════════════════════════════════
# STEP 4: Write results to CSV
# ══════════════════════════════════════════════════════

out_fields = ["borehole", "depth_m", "N60", "Vs_ms", "soil", "consistency", "saturated"]

with open("geotech_data/analysis_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=out_fields)
    writer.writeheader()
    writer.writerows(processed)

print("Step 4: ✓ Saved analysis_results.csv")
with open("geotech_data/analysis_results.csv", "r") as f:
    print(f.read())

In [ ]:
# ══════════════════════════════════════════════════════
# STEP 5: Write summary report (text file)
# ══════════════════════════════════════════════════════

all_n = [r["N60"] for r in processed]
all_vs = [r["Vs_ms"] for r in processed]

with open("geotech_data/summary_report.txt", "w") as f:
    f.write("=" * 50 + "\n")
    f.write("  SPT ANALYSIS SUMMARY REPORT\n")
    f.write("  BH-01 | Salt Lake City, Utah\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"  Total layers:    {len(processed)}\n")
    f.write(f"  Avg N60:         {sum(all_n)/len(all_n):.1f}\n")
    f.write(f"  Avg Vs:          {sum(all_vs)/len(all_vs):.1f} m/s\n")
    f.write(f"  Weak (N<15):     {len([n for n in all_n if n < 15])}\n")
    f.write(f"  Water table:     {wt}m\n\n")
    f.write("  Pipeline: raw_field_data.csv → analysis_results.csv\n")

print("Step 5: ✓ Saved summary_report.txt")
with open("geotech_data/summary_report.txt", "r") as f:
    print(f.read())

print("Complete pipeline: CSV in → Process → CSV out + TXT report")

---

## Practice Exercises

### Exercise 1: Write and read
Create a text file with 5 lines of borehole metadata. Read it back and print each line.

### Exercise 2: CSV round-trip
Write a CSV with columns `depth`, `N60`, `soil` for 5 layers. Read it back with `DictReader`, convert N60 to int, and compute the average.

### Exercise 3: Append log
Create a drilling log file. Append 3 morning entries, then 3 afternoon entries. Read and print the full log.

In [ ]:
# EXERCISE 1
# with open("geotech_data/exercise1.txt", "w") as f:
#     f.write("Project: ...\n")
#     ...

In [ ]:
# EXERCISE 2
# Write CSV, then read with DictReader, convert types, compute avg

In [ ]:
# EXERCISE 3
# Create with "w", append with "a" twice, read with "r"

---

## Key Takeaways

1. **Always use `with open(...) as f:`** — it auto-closes the file even if errors occur. Never use `open()`/`close()` manually.

2. **File modes:** `"r"` = read, `"w"` = write (overwrites!), `"a"` = append. Know the difference — `"w"` erases existing data!

3. **`csv.DictReader`** is the best way to read CSV — each row becomes a dictionary, access columns by name. Remember: all values are strings, convert with `int()` / `float()`.

4. **`csv.DictWriter`** writes list-of-dicts to CSV with zero manual formatting. This connects directly to Pandas: `pd.DataFrame(data).to_csv()`.

5. The real-world workflow: **CSV in → Process → CSV out + Report** — exactly what we built today.

---

## How This Connects to ML

| Today | ML Equivalent |
|-------|---------------|
| `csv.DictReader` | `pd.read_csv()` (Phase 2) |
| `csv.DictWriter` | `df.to_csv()` |
| Read → process → write | ETL pipeline (Extract-Transform-Load) |
| Text report | Model training logs, experiment tracking |
| Append mode | Logging training metrics per epoch |
| `os.path.exists` | Checking for saved model checkpoints |

---

*Day 008 of 100 — We can now read and write real data files!*